# 🐕 Damru AI — Phase 4: QLoRA Fine-Tuning

This Colab notebook fine-tunes a small open model on Damru's knowledge base (HuggingFace dataset `Damaru-ai/damru-knowledge`) to create Damru's OWN model.

**How to run:** Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> Run all.

⚠️ For good results you need a lot of Q&A (ideally 500+). Let the Phase 3 cron run for a while first. With few rows this is only a test run.

In [ ]:
# 1) Install Unsloth (fast QLoRA) — takes 2-3 min
!pip install -q unsloth

In [ ]:
# 2) Config
HF_DATASET = "Damaru-ai/damru-knowledge"   # unlimited free training data
HF_TOKEN   = ""   # optional: only needed to PUSH the trained model to HF
MODEL      = "unsloth/Qwen2.5-3B-Instruct"   # small, multilingual

In [ ]:
# 3) Load all Q&A from HuggingFace (unlimited free storage)
from datasets import load_dataset
hf = load_dataset(HF_DATASET, split="train")
rows = [{"question": x["question"], "answer": x["answer"]} for x in hf]
print("Loaded", len(rows), "Q&A pairs from HuggingFace")

In [ ]:
# 4) Build the training dataset (chat format)
from datasets import Dataset
SYS = "You are Damru, a helpful, multilingual AI assistant made by SHIVA AI. Answer clearly and reply in the same language the user uses."
conv = []
for x in rows:
    q = (x.get("question") or "").strip()
    a = (x.get("answer") or "").strip()
    if len(q) > 3 and len(a) > 20:
        conv.append({"conversations": [
            {"role": "system", "content": SYS},
            {"role": "user", "content": q},
            {"role": "assistant", "content": a},
        ]})
print("Usable examples:", len(conv))
dataset = Dataset.from_list(conv)

In [ ]:
# 5) Load a small model in 4-bit (QLoRA)
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL,
    max_seq_length = 2048,
    load_in_4bit = True,
    dtype = None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

In [ ]:
# 6) Apply the chat template
def to_text(ex):
    return {"text": tokenizer.apply_chat_template(ex["conversations"], tokenize=False, add_generation_prompt=False)}
dataset = dataset.map(to_text)
print(dataset[0]["text"][:500])

In [ ]:
# 7) Fine-tune (QLoRA). APIs change — if this errors, copy the trainer cell from Unsloth's latest Qwen notebook.
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 2048,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
    ),
)
trainer.train()

In [ ]:
# 8) Quick test
FastLanguageModel.for_inference(model)
msgs = [
    {"role": "system", "content": SYS},
    {"role": "user", "content": "Explain photosynthesis in simple words"},
]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.6)
print(tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
# 9) Save the model
# (a) LoRA adapters (small, local)
model.save_pretrained("damru-lora")
tokenizer.save_pretrained("damru-lora")

# (b) OPTIONAL: push merged 16-bit model to HuggingFace (needs HF_TOKEN)
# from huggingface_hub import login
# login(HF_TOKEN)
# model.push_to_hub_merged("Damaru-ai/damru-3b", tokenizer, save_method="merged_16bit", token=HF_TOKEN)

# (c) OPTIONAL: export GGUF for Ollama / llama.cpp
# model.save_pretrained_gguf("damru-gguf", tokenizer, quantization_method="q4_k_m")
print("Done. Damru model saved to ./damru-lora")

## ✅ Next: Phase 5 (Hybrid serving)

Once Damru's own model is trained and pushed to HuggingFace, Phase 5 wires it into the app as the PRIMARY brain, with the big free AIs kept as a fallback. The app keeps logging new Q&A, the Phase 3 cron keeps distilling, and you re-run this notebook every so often to make Damru smarter over time.